# Week 5 · Fundamentals of Azure AI

**The problem.** Lakeshore Ride's support inbox mixes billing disputes, broken docks and riders hurt on the trail. `POST /v1/classify` must label each message with a category and an urgency so the right team sees it first. Today it only knows the category names, and it guesses. By the end of this lab it follows rules you wrote, measured and deployed.

On the way you learn the one primitive every AI feature is built on: send text to a model, get text or data back.

| Part | What you do | Lecture notes |
|---|---|---|
| 1 | Where models live: account, project, deployment, endpoint | §2, §4 |
| 2 | First calls from scratch: tokens, reasoning effort, caching, state, streaming | §5, §6, §7 |
| 3 | Structured output: from free text to typed data | §8 |
| 4 | **Your turn:** measure the triage service, improve it with evidence | §9 |
| 5 | Deploy and score the live URL | |

**You ship:** a better `POST /v1/classify`, deployed. **You submit:** your URL.

**How to use this notebook.** Run the cells in order. Parts 1 to 3 show each idea with a small, runnable example; Part 4 is where you change the service; Part 5 ships it. The [lecture notes](lecture.md) explain the why behind each part.

**Optional, for depth:** [deep-dive.ipynb](deep-dive.ipynb) goes further (control plane, rate limits, conversations, reasoning effort measured, a Foundry agent). It is not graded and cannot change your score; open it after you submit.

Prerequisite: [Setup](../setup/README.md) is complete and your `.env` exists.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## Part 1 · Where models live

Microsoft Foundry organizes AI in four layers. You created all of them with Terraform in Setup.

| Layer | In your resource group | What it is for |
|---|---|---|
| Account (resource) | `aif-<you>-<suffix>` | Billing, network, identity. One endpoint host. |
| Project | `proj-<you>` | Your workspace: agents, evaluations, traces, connections. |
| Deployment | `gpt-5.4-mini`, `text-embedding-3-small` | A model you call **by name**, with its own capacity (tokens and requests per minute). |
| Endpoint | `https://aif-….openai.azure.com/openai/v1/` | The URL your code calls. The v1 API has the same shape as OpenAI's. |

Your code never names a model version, only a **deployment**. Swapping the model behind a deployment changes nothing in the code.

In [ ]:
out = course.outputs()
for key in ["foundry_account_name", "foundry_openai_endpoint", "foundry_project_endpoint", "chat_deployment", "embedding_deployment"]:
    print(f"{key:26s} {out[key]}")

course.run([
    "az", "cognitiveservices", "account", "deployment", "list",
    "-n", out["foundry_account_name"], "-g", out["resource_group_name"],
    "--query", "[].{deployment:name, model:properties.model.name, version:properties.model.version, capacity:sku.capacity}",
    "-o", "table",
])

`capacity` is in thousands of tokens per minute, and it also sets a requests-per-minute limit: `capacity = 100` gives 100,000 tokens and 100 requests a minute. Short calls such as triage hit the request limit first. Go over either and you get HTTP 429 (too many requests); a well-behaved client waits and retries.

## Part 2 · First calls, from scratch

No helpers this time. Build the client yourself so you see every piece:

- `DefaultAzureCredential` finds an identity: your `az login` on this laptop, the managed identity in Azure.
- `get_bearer_token_provider` turns it into a function that returns a fresh token for the Foundry audience.
- The OpenAI SDK sends that token instead of an API key. There is no key to leak.

In [ ]:
import os
import time

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")
client = OpenAI(base_url=os.environ["FOUNDRY_OPENAI_ENDPOINT"], api_key=token_provider)
MODEL = os.environ["FOUNDRY_MODEL"]

start = time.perf_counter()
response = client.responses.create(
    model=MODEL,
    instructions="You are the support assistant for Lakeshore Ride, a bike-share service. Answer in one sentence.",
    input="Can I end my ride at any station?",
)
print(response.output_text)
print(f"\n{time.perf_counter() - start:.1f} s | tokens in {response.usage.input_tokens}, out {response.usage.output_tokens}")

Two inputs, two roles:

- `instructions` say **who the model is and how to behave**. You write them once, in code.
- `input` is **what the user said**. It changes on every request, and you never trust it.

The model knows nothing about Lakeshore Ride: it answered from general knowledge. In Week 6 you give it the handbook.

### What you pay for

You pay per token, input and output, each at its own price. A token is about four characters of English. Reasoning models can also think before they answer: **reasoning tokens**, which you do not see but do pay for, billed as output. The `reasoning` setting is the dial. For `gpt-5.4-mini` it accepts `none` (the default), `low`, `medium`, `high` and `xhigh`. One consequence: sampling settings such as `temperature` work only with `none`; with any other effort the API rejects them. Compare:

In [ ]:
question = "A member rides an e-bike for 40 minutes at $0.18 per minute. What do they pay? Answer with the amount only."

for effort in ["none", "low", "medium", "high", "xhigh"]:
    start = time.perf_counter()
    r = client.responses.create(model=MODEL, input=question, reasoning={"effort": effort})
    reasoning_tokens = r.usage.output_tokens_details.reasoning_tokens
    print(f"{effort:7s} {time.perf_counter() - start:5.1f} s  output {r.usage.output_tokens:4d} tokens "
          f"(reasoning {reasoning_tokens:4d})  -> {r.output_text}")

With `none` the model answers at once. On an easy question every setting agrees, so `none` is the fastest and cheapest. On hard ones (several rules at once, multi-step arithmetic) more effort can change the answer. You decide per task, by measurement.

### Prompt caching

Most requests repeat a long prefix: instructions, tool definitions, examples. The service caches it. When the first 1,024 or more tokens of a request match a recent request, they count as **cached input**: cheaper and faster. You get this for free if you put the stable part first and the variable part (the user's message) last. Here the whole handbook goes in the instructions; watch `cached`:

In [ ]:
handbook = "\n\n".join(page.read_text() for page in sorted(Path("app/data/handbook").glob("*.md")))
policy = "You are the Lakeshore Ride support assistant. Answer in one sentence, from this handbook only.\n\n" + handbook

for question in ["Can I bring my dog on a bike?", "How long is a day pass valid?", "Is there a student plan?"]:
    r = client.responses.create(model=MODEL, instructions=policy, input=question, prompt_cache_key="lakeshore-handbook")
    print(f"input {r.usage.input_tokens:5d} tokens, cached {r.usage.input_tokens_details.cached_tokens:5d}  | {question}")

The first call pays for the whole handbook; the next ones find it in the cache. Two rules follow. Put stable content first. Never put something that changes on every request (a timestamp, a request id) at the top of your instructions: it breaks the match for everything after it. Next week you meet the alternative to sending the whole handbook every time: retrieve only the passages a question needs.

### Conversation state

The model has no memory. Each call stands alone unless you connect it. With the Responses API you pass `previous_response_id`, and the service keeps the history for you.

In [ ]:
first = client.responses.create(model=MODEL, input="My name is Ada and I ride a classic bike.")
second = client.responses.create(model=MODEL, input="What is my name, and which kind of bike do I ride?")
third = client.responses.create(
    model=MODEL,
    input="What is my name, and which kind of bike do I ride?",
    previous_response_id=first.id,
)
print("without state:", second.output_text)
print("with state:   ", third.output_text)

### Streaming

A long answer can take several seconds. Streaming sends it while it is produced, so a user sees the first words almost at once. The SDK yields **events**; the text arrives in `response.output_text.delta` events.

In [ ]:
start = time.perf_counter()
first_words = None
for event in client.responses.create(model=MODEL, input="Give three tips for a safe first bike-share ride.", stream=True):
    if event.type == "response.output_text.delta":
        first_words = first_words or time.perf_counter() - start
        print(event.delta, end="", flush=True)
print(f"\n\nfirst words after {first_words:.1f} s, complete after {time.perf_counter() - start:.1f} s")

## Part 3 · Structured output

A service cannot parse prose. Asking politely for JSON works most of the time, which in production means it fails some of the time. **Structured output** gives the model a JSON schema, and the API guarantees the response matches it.

You describe the schema with a Pydantic model. This is the one the triage service uses, from `app/contract.py`:

In [ ]:
import inspect
import json

from app.contract import TicketLabel

print(inspect.getsource(TicketLabel))
print(json.dumps(TicketLabel.model_json_schema()["properties"]["category"], indent=2))

In [ ]:
ticket = "The dock at LS-10 won't lock my bike and I can't end my ride. It's been 20 minutes."

parsed = client.responses.parse(
    model=MODEL,
    instructions="Classify this Lakeshore Ride support ticket.",
    input=ticket,
    text_format=TicketLabel,
)
label = parsed.output_parsed
print(type(label).__name__, "->", label)

`label` is a typed Python object, not a string. If the model tried to invent a category, the API would not let it.

The schema guarantees the **shape**, not the **judgment**. Is that ticket `station_issue` or `bike_issue`? Is it `high` urgency because the rider is stuck right now? The model cannot know your business rules unless you write them down. That is Part 4.

## Part 4 · Measure, then improve

The service's triage lives in `app/services/classify.py`. Here are its instructions today:

In [ ]:
from app.services import classify

print(classify.INSTRUCTIONS)

The model sees the task and nothing else: no definitions, no rules. Measure before you change anything. `in_process=True` runs your service from the files in `app/` without deploying it. The harness sends the 24 public tickets and compares each label with the expected one.

In [ ]:
baseline = course.check(5, in_process=True)

Read the misses, not just the score. For each one ask: *what rule would a new support agent need to get this right?*

In [ ]:
rows = [json.loads(line) for line in Path("harness/data/week05.jsonl").read_text().splitlines() if line.strip()]
tickets = {row["id"]: row for row in rows}

for check in baseline["checks"]:
    if check["id"] in ("w05.classify.category", "w05.classify.urgency"):
        print(f"{check['id']}: {check['detail']}\n")

for tid in ["t04", "t11", "t14", "t23"]:
    t = tickets[tid]
    print(f"{tid} expected {t['category']}/{t['urgency']}: {t['text']}")

### Your turn

Open `app/services/classify.py` and rewrite `INSTRUCTIONS`. A good triage policy has:

1. **A definition for every category**, with what belongs in it and what does not. For example, `billing` is a charge, refund or credit the rider disputes; a question about what a pass costs, with nothing charged, is not billing.
2. **Urgency rules a person could apply**: *high* when someone is hurt or at risk, or the rider is stuck right now; *medium* when money is involved or a bike or station is unusable; *low* otherwise.
3. **Tie-breakers** for tickets that fit two categories. Safety beats everything. A refund request is billing, even if a broken bike caused it.
4. **No ticket ids or exact sentences from the public data.** We grade on tickets you have not seen. Write rules, not answers.

Small models read instructions literally. Put the rules that matter most first, say what to do when two rules conflict, and add one worked example rather than ten adjectives ([prompting GPT-5.4 models, OpenAI](https://developers.openai.com/api/docs/guides/latest-model/gpt-5.4)). Aim for the right altitude: specific enough to decide a case, general enough to cover tickets you have not seen ([context engineering, Anthropic](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)).

Save the file and run the next cell. The harness starts a fresh process, so it always sees your latest code. After each run, explain to your neighbor *why* each rule helped.

**100 is reachable.** Every public ticket can be labelled by a rule a support agent could follow, and each miss in the report tells you which rule is missing. For each miss, write the general rule (never the ticket), re-run, and check that no other ticket broke. A ticket that flips between runs has an ambiguous rule: make it explicit, or add one short worked example. A ticket that comes back invalid (a 500 in the proof) was refused by the model: say in the instructions that every ticket gets a label, including injuries and threats, because support must see those first. In class, stop at the 45-minute mark and deploy your best version (Part 5): a deployed 90 beats an undeployed 95. Then keep going until the local harness shows 100, and redeploy before the deadline.

In [ ]:
improved = course.check(5, in_process=True)
print(f"\nbaseline {baseline['score']}  ->  now {improved['score']}")

Models are not deterministic: the same prompt can score a point or two differently between runs. Treat a change of one point as noise and five points as signal, and run twice before you trust a 100.

## Part 5 · Deploy and score the live URL

The same two steps every week. `deploy()` builds the image in your registry, rolls out a new revision of the Container App, and waits until the new version answers. Then the harness scores the live URL: that is what we grade.

In [ ]:
url = course.deploy()
proof = course.check(5, url)

### Submit

Submit the URL printed above by the deadline in the [Week 5 README](README.md). Keep the proof in `reports/`: if your grade surprises you, it shows exactly what your service answered.

### Go further

- Add three or four **few-shot examples** to the instructions. Which tickets do they fix? Which do they break?
- In `app/models.py`, pass `reasoning={"effort": "low"}` to `responses.parse`. Does accuracy move? Latency? Cost?
- Add a second deployment, `gpt-5.4-nano`, in `infra/foundry.tf` and compare accuracy, latency and price. Or deploy the [model router](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router), which picks a model per request.
- Foundry also serves models from other providers. Claude models use Anthropic's Messages API, not the Responses API, with the same Entra ID token: read [Claude in Microsoft Foundry](https://platform.claude.com/docs/en/build-with-claude/claude-in-microsoft-foundry) and compare one ticket across both.
- **Deep dive (optional):** [deep-dive.ipynb](deep-dive.ipynb) reads your deployment's limits from the control plane, takes a token apart, measures reasoning effort on these tickets, and builds a versioned **Foundry prompt agent**.

### What to remember

- Code calls a **deployment** by name; Entra ID tokens replace API keys.
- You pay per token, reasoning tokens included. Effort is a dial (`none` by default on `gpt-5.4-mini`): measure where to set it.
- A stable prefix is cached: keep instructions first and the user's message last.
- Structured output fixes the **shape**. Instructions carry the **judgment**: business rules belong there, written as rules.
- Change one thing, measure, and keep it only if the score moves for a reason you can explain.

**Learn more:** [Develop a generative AI chat app with Microsoft Foundry](https://learn.microsoft.com/en-us/training/modules/foundry-sdk/) · [Get started with Microsoft Foundry (code)](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code) · [Structured model outputs (OpenAI)](https://developers.openai.com/api/docs/guides/structured-outputs) · [Reasoning models (OpenAI)](https://developers.openai.com/api/docs/guides/reasoning) · [Prompt caching (OpenAI)](https://developers.openai.com/api/docs/guides/prompt-caching)